# including calculating age column, just experiment.
# result : the age column did not affect the final scores for this data set.
## but for best practice it is better to calculate it rather than leave the built year as it.
#### You can see the changes below.

In [9]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [10]:
df = pd.read_csv('car_prices.csv')

In [11]:
df_modified = df.drop(columns=['vin', 'mmr', 'seller','make'])

In [12]:
## missing values in target must drop
df_modified['sellingprice'].isna().sum()

12

In [13]:
df_modified = df_modified.dropna(subset=['sellingprice'])
df_modified['sellingprice'].isna().sum()

0

In [14]:
bins = [1980 , 1990 , 1995, 2000, 2005, 2010, np.inf]
labels =[1, 2, 3, 4, 5, 6]
df_modified['binned_year'] = pd.cut(df_modified['year'], bins=bins, labels=labels)

In [15]:
inputs = df_modified.drop(columns=['sellingprice'])
outputs = df_modified['sellingprice']

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(inputs,outputs, test_size=0.1, stratify=df_modified['binned_year'], random_state=42)

In [17]:
X_train = X_train.drop(columns=['binned_year'])
X_test = X_test.drop(columns=['binned_year'])

In [18]:
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((502942, 11), (55883, 11), (502942,), (55883,))

In [19]:
import importlib
import custom_transformers

# Force the kernel to read the file again
importlib.reload(custom_transformers)

<module 'custom_transformers' from 'C:\\Users\\ASUS\\Desktop\\ML\\USA cars prices\\custom_transformers.py'>

In [20]:
from sklearn.preprocessing import FunctionTransformer
    
log_transformer = FunctionTransformer(np.log1p, feature_names_out='one-to-one')

In [22]:
from custom_transformers import NumericStringToNaN
nn = NumericStringToNaN()

In [23]:
from custom_transformers import LowerCase
lc = LowerCase()

In [24]:
from custom_transformers import DashToNaN
dn = DashToNaN()


In [25]:
from custom_transformers import YearExtractor
ye = YearExtractor()


In [26]:
from custom_transformers import KeepAutoManual
km = KeepAutoManual()

In [27]:
from custom_transformers import LongStringToNaN
ls = LongStringToNaN()

In [28]:
from custom_transformers import FrequencyEncoder


In [29]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder,StandardScaler

In [33]:
num_pipeline = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='median')),
    ('transformer' , FunctionTransformer(np.log1p, feature_names_out='one-to-one' )),
    ('scaler' , StandardScaler())
])
num_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('transformer',
                 FunctionTransformer(feature_names_out='one-to-one',
                                     func=<ufunc 'log1p'>)),
                ('scaler', StandardScaler())])

In [44]:
trim_pipeline =  Pipeline(steps=[
    ('transformer1' , DashToNaN()),
    ('imputer' , SimpleImputer(strategy='constant', fill_value='unknown')),
    ('transformer2' , LowerCase()),
    ('encoder' , FrequencyEncoder())
])
trim_pipeline

Pipeline(steps=[('transformer1', DashToNaN()),
                ('imputer',
                 SimpleImputer(fill_value='unknown', strategy='constant')),
                ('transformer2', LowerCase()),
                ('encoder', FrequencyEncoder())])

In [50]:
model_pipeline =  Pipeline(steps=[
    ('transformer1' , DashToNaN()),
    ('imputer' , SimpleImputer(strategy='constant', fill_value='unknown')),
    ('transformer2' , LowerCase()),
    ('encoder' , TargetEncoder(target_type='continuous' , cv=5))
])
model_pipeline

Pipeline(steps=[('transformer1', DashToNaN()),
                ('imputer',
                 SimpleImputer(fill_value='unknown', strategy='constant')),
                ('transformer2', LowerCase()),
                ('encoder', TargetEncoder(target_type='continuous'))])

In [51]:
body_pipeline =  Pipeline(steps=[
    ('transformer1' , DashToNaN()),
    ('imputer' , SimpleImputer(strategy='constant', fill_value='unknown')),
    ('transformer2' , LowerCase()),
    ('encoder' , OneHotEncoder(handle_unknown='ignore' , sparse_output=False))
])
body_pipeline

Pipeline(steps=[('transformer1', DashToNaN()),
                ('imputer',
                 SimpleImputer(fill_value='unknown', strategy='constant')),
                ('transformer2', LowerCase()),
                ('encoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [52]:
state_pipeline =  Pipeline(steps=[
    ('transformer1' , DashToNaN()),
    ('transformer2' , LongStringToNaN()),
    ('imputer' , SimpleImputer(strategy='constant', fill_value='unknown')),
    ('transformer3' , LowerCase()),
   
    ('encoder' , OneHotEncoder(handle_unknown='ignore' , sparse_output=False))
])
state_pipeline

Pipeline(steps=[('transformer1', DashToNaN()),
                ('transformer2', LongStringToNaN()),
                ('imputer',
                 SimpleImputer(fill_value='unknown', strategy='constant')),
                ('transformer3', LowerCase()),
                ('encoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [53]:
colors_pipeline =  Pipeline(steps=[
    ('transformer1' , DashToNaN()),
    ('transformer2' , NumericStringToNaN()),
    ('imputer' , SimpleImputer(strategy='constant', fill_value='unknown')),
    ('transformer3' , LowerCase()),
    ('scaler' , FrequencyEncoder())
])
colors_pipeline

Pipeline(steps=[('transformer1', DashToNaN()),
                ('transformer2', NumericStringToNaN()),
                ('imputer',
                 SimpleImputer(fill_value='unknown', strategy='constant')),
                ('transformer3', LowerCase()), ('scaler', FrequencyEncoder())])

In [56]:
transmission_pipeline =  Pipeline(steps=[
    ('transformer1' , DashToNaN()),
    ('transformer2' , KeepAutoManual()),
    ('transformer3' , LowerCase()),
    ('imputer' , SimpleImputer(strategy='most_frequent')),
    ('encoder' , OneHotEncoder(handle_unknown='ignore' , sparse_output=False))
])
transmission_pipeline

Pipeline(steps=[('transformer1', DashToNaN()),
                ('transformer2', KeepAutoManual()),
                ('transformer3', LowerCase()),
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [58]:
date_sold_pipeline =  Pipeline(steps=[
    ('transformer1' , DashToNaN()),
    ('transformer2' , YearExtractor()),
    ('imputer' , SimpleImputer(strategy='most_frequent')),
   ('scaler' ,StandardScaler())
])
date_sold_pipeline

Pipeline(steps=[('transformer1', DashToNaN()),
                ('transformer2', YearExtractor()),
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('scaler', StandardScaler())])

In [63]:
year_built_pipeline = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='most_frequent')),
    ('transformer' , FunctionTransformer(np.log1p, feature_names_out='one-to-one' )),
    ('scaler' , StandardScaler())
])
year_built_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('transformer',
                 FunctionTransformer(feature_names_out='one-to-one',
                                     func=<ufunc 'log1p'>)),
                ('scaler', StandardScaler())])

# From here the pipeline changes from the original one

In [70]:
from sklearn.compose import ColumnTransformer

In [72]:
age_date_sold_pipeline =  Pipeline(steps=[
    ('transformer1' , DashToNaN()),
    ('transformer2' , YearExtractor()),
    ('imputer' , SimpleImputer(strategy='most_frequent')),
])
age_date_sold_pipeline

Pipeline(steps=[('transformer1', DashToNaN()),
                ('transformer2', YearExtractor()),
                ('imputer', SimpleImputer(strategy='most_frequent'))])

In [74]:
age_year_built_pipeline = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='most_frequent')),
])
age_year_built_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))])

In [77]:
age_preprocessing = ColumnTransformer(transformers=[
    ('age_year_built' , age_year_built_pipeline , ['year']),
    ('age_date_sold' , age_date_sold_pipeline, ['saledate'])
])
    
age_preprocessing

ColumnTransformer(transformers=[('age_year_built',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['year']),
                                ('age_date_sold',
                                 Pipeline(steps=[('transformer1', DashToNaN()),
                                                 ('transformer2',
                                                  YearExtractor()),
                                                 ('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['saledate'])])

In [79]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

class AgeCalculator(BaseEstimator, TransformerMixin):
    def __init__(self, year_built_idx=0, year_sold_idx=1):
        """
        Takes the column indices to allow flexibility if your upstream 
        ColumnTransformer changes order or outputs multiple columns.
        """
        self.year_built_idx = year_built_idx
        self.year_sold_idx = year_sold_idx

    def fit(self, X, y=None):
        # Stateless transformer, so fit just returns self
        return self

    def transform(self, X):
        # Convert to numpy array in case X is a pandas DataFrame or sparse matrix
        X = np.asarray(X)
        
        # Vectorized C-speed subtraction
        age = X[:, self.year_sold_idx] - X[:, self.year_built_idx]
        
        # Scikit-learn requires 2D array outputs for transformers
        return age.reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(['property_age'])

In [81]:
age_pipeline =  Pipeline(steps=[
    ('preprocessing_separetly' , age_preprocessing),
    ('age_transformer' , AgeCalculator()),
   ('scaler' ,StandardScaler())
])
age_pipeline

Pipeline(steps=[('preprocessing_separetly',
                 ColumnTransformer(transformers=[('age_year_built',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent'))]),
                                                  ['year']),
                                                 ('age_date_sold',
                                                  Pipeline(steps=[('transformer1',
                                                                   DashToNaN()),
                                                                  ('transformer2',
                                                                   YearExtractor()),
                                                                  ('imputer',
                                                                   SimpleImputer(strategy='most_frequent'))]),
                                                  ['saledate'])])),
                ('age_transformer', AgeCalculator()),
                ('scaler', StandardScaler())])

In [83]:
temp = age_pipeline.fit_transform(X_train[['year','saledate']])

In [84]:
temp[:10]

array([[ 0.28809478],
       [-0.21904076],
       [-0.97974407],
       [-0.97974407],
       [ 2.06306916],
       [-0.47260853],
       [-0.97974407],
       [-0.21904076],
       [ 0.03452701],
       [-0.47260853]])

In [87]:
num_features = ['condition','odometer']

In [89]:
from sklearn.compose import ColumnTransformer

In [91]:
preprocessing = ColumnTransformer(transformers=[
    ('age' , age_pipeline, ['year','saledate']),
    ('date_sold' , date_sold_pipeline, ['saledate']),
    ('transmission_encoded' , transmission_pipeline , ['transmission']),
    ('colores_encoded' , colors_pipeline , ['color', 'interior']),
    ('state_encoded' , state_pipeline , ['state']),
    ('body_encoded' , body_pipeline , ['body']),
    ('model_encoded' , model_pipeline , ['model']),
    ('trim_encoded' , trim_pipeline , ['trim']),
    ('logged' , num_pipeline , num_features)])

preprocessing

ColumnTransformer(transformers=[('age',
                                 Pipeline(steps=[('preprocessing_separetly',
                                                  ColumnTransformer(transformers=[('age_year_built',
                                                                                   Pipeline(steps=[('imputer',
                                                                                                    SimpleImputer(strategy='most_frequent'))]),
                                                                                   ['year']),
                                                                                  ('age_date_sold',
                                                                                   Pipeline(steps=[('transformer1',
                                                                                                    DashToNaN()),
                                                                                                   ('transformer2',
                                                                                                    YearExtractor()),
                                                                                                   ('imputer',
                                                                                                    SimpleImputer(strategy='most_fre...
                                                 ('imputer',
                                                  SimpleImputer(fill_value='unknown',
                                                                strategy='constant')),
                                                 ('transformer2', LowerCase()),
                                                 ('encoder',
                                                  FrequencyEncoder())]),
                                 ['trim']),
                                ('logged',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('transformer',
                                                  FunctionTransformer(feature_names_out='one-to-one',
                                                                      func=<ufunc 'log1p'>)),
                                                 ('scaler', StandardScaler())]),
                                 ['condition', 'odometer'])])

In [93]:
preprocessing.set_output(transform="pandas")

ColumnTransformer(transformers=[('age',
                                 Pipeline(steps=[('preprocessing_separetly',
                                                  ColumnTransformer(transformers=[('age_year_built',
                                                                                   Pipeline(steps=[('imputer',
                                                                                                    SimpleImputer(strategy='most_frequent'))]),
                                                                                   ['year']),
                                                                                  ('age_date_sold',
                                                                                   Pipeline(steps=[('transformer1',
                                                                                                    DashToNaN()),
                                                                                                   ('transformer2',
                                                                                                    YearExtractor()),
                                                                                                   ('imputer',
                                                                                                    SimpleImputer(strategy='most_fre...
                                                 ('imputer',
                                                  SimpleImputer(fill_value='unknown',
                                                                strategy='constant')),
                                                 ('transformer2', LowerCase()),
                                                 ('encoder',
                                                  FrequencyEncoder())]),
                                 ['trim']),
                                ('logged',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('transformer',
                                                  FunctionTransformer(feature_names_out='one-to-one',
                                                                      func=<ufunc 'log1p'>)),
                                                 ('scaler', StandardScaler())]),
                                 ['condition', 'odometer'])])

## these hyper params. are for the pipeline without the age calculation, only experiment!!

In [98]:
from xgboost import XGBRegressor

xgb_model = Pipeline(steps=[
    ('preprocessing' , preprocessing),
    ('xgb_reg' , XGBRegressor(random_state = 42))])

xgb_model

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('age',
                                                  Pipeline(steps=[('preprocessing_separetly',
                                                                   ColumnTransformer(transformers=[('age_year_built',
                                                                                                    Pipeline(steps=[('imputer',
                                                                                                                     SimpleImputer(strategy='most_frequent'))]),
                                                                                                    ['year']),
                                                                                                   ('age_date_sold',
                                                                                                    Pipeline(steps=[('transformer1',
                                                                                                                     DashToNaN()),
                                                                                                                    ('transformer2',
                                                                                                                     YearExtractor()),
                                                                                                                    ('imputer'...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=None, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [141]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'xgb_reg__n_estimators': randint(300,700),
    'xgb_reg__learning_rate': [0.01, 0.05, 0.09,0.1],
    'xgb_reg__max_depth':  randint(5,16),
    'xgb_reg__colsample_bytree': [0.3,0.4,0.5],
    'xgb_reg__min_child_weight': randint(4,30),
}



xgb_search = RandomizedSearchCV(
    xgb_model, 
    param_distributions=param_dist,
    n_iter=30, 
    cv=3, 
    scoring='neg_root_mean_squared_error',
    n_jobs=15, 
    verbose=3
)

xgb_search.fit(X_train, Y_train)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('age',
                                                                               Pipeline(steps=[('preprocessing_separetly',
                                                                                                ColumnTransformer(transformers=[('age_year_built',
                                                                                                                                 Pipeline(steps=[('imputer',
                                                                                                                                                  SimpleImputer(strategy='most_frequent'))]),
                                                                                                                                 ['year']),
                                                                                                                                ('age_date_sold',
                                                                                                                                 Pipeline(steps=[('transformer1',
                                                                                                                                                  DashToNaN()),
                                                                                                                                                 ('transfo...
                                        'xgb_reg__max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000001972374E7E0>,
                                        'xgb_reg__min_child_weight': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000001972374C8F0>,
                                        'xgb_reg__n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x000001971B688050>},
                   scoring='neg_root_mean_squared_error', verbose=3)

In [147]:
xgb_model = xgb_search.best_estimator_

In [149]:
xgb_search.best_params_

{'xgb_reg__colsample_bytree': 0.5,
 'xgb_reg__learning_rate': 0.1,
 'xgb_reg__max_depth': 13,
 'xgb_reg__min_child_weight': 11,
 'xgb_reg__n_estimators': 493}

In [151]:
from sklearn.model_selection import cross_validate
import numpy as np

scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mape': 'neg_mean_absolute_percentage_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

cv_results_xgb = cross_validate(
    xgb_model, 
    X_train, Y_train, 
    cv=3, 
    n_jobs=15,
    verbose=3,
    scoring=scoring,
    return_train_score=True
)

[Parallel(n_jobs=15)]: Using backend LokyBackend with 15 concurrent workers.
[Parallel(n_jobs=15)]: Done   3 out of   3 | elapsed:  1.6min finished


In [153]:
-cv_results_xgb['train_rmse'].mean(), -cv_results_xgb['test_rmse'].mean()

(1521.5351408155375, 2346.767497179203)

In [155]:
-cv_results_xgb['train_mae'].mean(), -cv_results_xgb['test_mae'].mean()

(933.9090016863743, 1273.1484456624792)

In [157]:
-cv_results_xgb['train_mape'].mean(), -cv_results_xgb['test_mape'].mean()

(0.1686264343275187, 0.24095107030735255)

In [159]:
cv_results_xgb['train_r2'].mean(), cv_results_xgb['test_r2'].mean()

(0.9756236197620493, 0.9419942558226694)

## V2

In [165]:
from xgboost import XGBRegressor

xgb_model_v2 = Pipeline(steps=[
    ('preprocessing' , preprocessing),
    ('xgb_reg' , XGBRegressor(random_state = 42))
])

xgb_model_v2

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('age',
                                                  Pipeline(steps=[('preprocessing_separetly',
                                                                   ColumnTransformer(transformers=[('age_year_built',
                                                                                                    Pipeline(steps=[('imputer',
                                                                                                                     SimpleImputer(strategy='most_frequent'))]),
                                                                                                    ['year']),
                                                                                                   ('age_date_sold',
                                                                                                    Pipeline(steps=[('transformer1',
                                                                                                                     DashToNaN()),
                                                                                                                    ('transformer2',
                                                                                                                     YearExtractor()),
                                                                                                                    ('imputer'...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=None, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [169]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'xgb_reg__n_estimators': [493],
    'xgb_reg__learning_rate': [0.1],
    'xgb_reg__max_depth':  [8,9,10,11],
    'xgb_reg__colsample_bytree': [ 0.5],
    'xgb_reg__min_child_weight': [11],
}


grid_search_v2 = GridSearchCV(
    xgb_model_v2, 
    param_grid, 
    cv=3,                  
    scoring='neg_root_mean_squared_error',
    n_jobs=15,             
    verbose=3,
    return_train_score=True
)

grid_search_v2.fit(X_train,Y_train)

Fitting 3 folds for each of 4 candidates, totalling 12 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('age',
                                                                         Pipeline(steps=[('preprocessing_separetly',
                                                                                          ColumnTransformer(transformers=[('age_year_built',
                                                                                                                           Pipeline(steps=[('imputer',
                                                                                                                                            SimpleImputer(strategy='most_frequent'))]),
                                                                                                                           ['year']),
                                                                                                                          ('age_date_sold',
                                                                                                                           Pipeline(steps=[('transformer1',
                                                                                                                                            DashToNaN()),
                                                                                                                                           ('transformer2'...
                                                     monotone_constraints=None,
                                                     multi_strategy=None,
                                                     n_estimators=None,
                                                     n_jobs=None,
                                                     num_parallel_tree=None, ...))]),
             n_jobs=15,
             param_grid={'xgb_reg__colsample_bytree': [0.5],
                         'xgb_reg__learning_rate': [0.1],
                         'xgb_reg__max_depth': [8, 9, 10, 11],
                         'xgb_reg__min_child_weight': [11],
                         'xgb_reg__n_estimators': [493]},
             return_train_score=True, scoring='neg_root_mean_squared_error',
             verbose=3)

In [171]:
xgb_model_v2 = grid_search_v2.best_estimator_

In [173]:
grid_search_v2.best_params_

{'xgb_reg__colsample_bytree': 0.5,
 'xgb_reg__learning_rate': 0.1,
 'xgb_reg__max_depth': 11,
 'xgb_reg__min_child_weight': 11,
 'xgb_reg__n_estimators': 493}

In [175]:
from sklearn.model_selection import cross_validate
import numpy as np

scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mape': 'neg_mean_absolute_percentage_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

cv_results_xgb_v2 = cross_validate(
    xgb_model_v2, 
    X_train, Y_train, 
    cv=3, 
    n_jobs=15,
    verbose=3,
    scoring=scoring,
    return_train_score=True
)

[Parallel(n_jobs=15)]: Using backend LokyBackend with 15 concurrent workers.
[Parallel(n_jobs=15)]: Done   3 out of   3 | elapsed:  1.3min finished


In [177]:
-cv_results_xgb_v2['train_rmse'].mean(), -cv_results_xgb_v2['test_rmse'].mean()

(1717.8374075609372, 2369.903631398796)

In [179]:
-cv_results_xgb_v2['train_mae'].mean(), -cv_results_xgb_v2['test_mae'].mean()

(1055.779460138064, 1292.6823014975116)

In [181]:
-cv_results_xgb_v2['train_mape'].mean(), -cv_results_xgb_v2['test_mape'].mean()

(0.1865082637764642, 0.24606284095375822)

In [183]:
cv_results_xgb_v2['train_r2'].mean(), cv_results_xgb_v2['test_r2'].mean()

(0.9689243482218646, 0.9408650911438255)

In [188]:
import joblib

In [ ]:
joblib.dump(xgb_model_v2 , 'age calculated, not best')